# HVAC 冷热切换核验
固定第 1900 回合模型，55 天确定性回放，48 状态反事实抽样；未重训练。完整范围和限制见 README.md。

In [ ]:
from pathlib import Path
import json, pandas as pd, numpy as np
audit_dir = next(root / 'reports/hvac_audit_20261007' for root in [Path.cwd(), *Path.cwd().parents] if (root / 'reports/hvac_audit_20261007/audit_summary.json').exists())
summary = json.loads((audit_dir / 'audit_summary.json').read_text(encoding='utf-8'))
summary['scope'], summary['best_episode'], summary['checks']

In [ ]:
pd.DataFrame(summary['baseline']).query('threshold_kw == 1')

In [ ]:
probes = pd.read_csv(audit_dir / 'counterfactual_probes.csv')
local = probes[np.isclose(abs(probes.candidate_kw - probes.reference_kw), 1, atol=1e-6, rtol=0)]
meaningful = local[local.gain.abs() > .001]
assert len(local) == 95 and len(meaningful) == 51
{'Q1_wrong': int(sum(meaningful.dq1 * meaningful.gain < 0)), 'both_wrong': int(sum((meaningful.dq1 * meaningful.gain < 0) & (meaningful.dq2 * meaningful.gain < 0))), 'actual_median_abs_gain': local.gain.abs().median(), 'Q1_median_abs_error': (local.dq1 - local.gain).abs().median()}

In [ ]:
pd.DataFrame(summary['deadband'])

In [ ]:
cost_columns = [c for c in probes if c.startswith('delta_cost_')]
assert np.max(abs(probes.gain + probes[cost_columns].sum(axis=1) / 100)) < 1e-10
summary['caveats']